# Investigate and setup paths

In [1]:
from pathlib import Path

In [2]:
print(Path.cwd())

/Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks


In [3]:
for path in sorted(Path.cwd().iterdir()): 
    print(" -", path)to

 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/.DS_Store
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/.ipynb_checkpoints
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/data
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/data_collection.ipynb
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/evals
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/evals.ipynb
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/mlops_prep_llm_training_serving.ipynb
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/cour

In [4]:
config_path = Path("../configs/config.yaml")

# Read the configs

In [5]:
import yaml

In [6]:
with config_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

In [7]:
print(type(config))

<class 'dict'>


In [8]:
print(config)

{'model': {'name': 'Qwen/Qwen2.5-1.5B'}, 'data': {'path': 'data/01_cold_start_cot_sft/data.jsonl'}, 'output': {'directory': 'models/adapter_experiment'}, 'peft': {'method': 'lora', 'r': 16, 'alpha': 32, 'dropout': 0.05, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']}, 'training': {'num_train_epochs': 3, 'batch_size': 4, 'gradient_accumulation_steps': 4, 'learning_rate': 0.1002, 'max_length': 1024, 'seed': 42}}


In [9]:
from pprint import pprint

In [10]:
print("Model config:")
pprint(config["model"], sort_dicts=False)

Model config:
{'name': 'Qwen/Qwen2.5-1.5B'}


## Reading configs in variables

In [11]:
model_name = config["model"]["name"]
data_path = config["data"]["path"]
peft_method = config["peft"]["method"]
learning_rate = config["training"]["learning_rate"]

In [12]:
print("Model:", model_name)
print("Data:", data_path)
print("PEFT method:", peft_method)
print("Learning rate:", learning_rate)

Model: Qwen/Qwen2.5-1.5B
Data: data/01_cold_start_cot_sft/data.jsonl
PEFT method: lora
Learning rate: 0.1002


In [13]:
values_to_check = {
    "model_name": model_name,
    "data_path": data_path,
    "peft_method": peft_method,
    "learning_rate": learning_rate,
    "target_modules": config["peft"]["target_modules"],
}

In [14]:
for name, value in values_to_check.items():
    print(f"{name}: {value!r} | type={type(value).__name__}")

model_name: 'Qwen/Qwen2.5-1.5B' | type=str
data_path: 'data/01_cold_start_cot_sft/data.jsonl' | type=str
peft_method: 'lora' | type=str
learning_rate: 0.1002 | type=float
target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'] | type=list


## Validation configuations 

In [15]:
assert isinstance(config, dict)
assert config["model"]["name"]
assert config["peft"]["method"] in {"lora", "adalora", "ia3"}
assert config["training"]["learning_rate"] > 0
assert isinstance(config["peft"]["target_modules"], list)


# Connecting the above config to the existing TrainConfig class

## data and path shenanigans

In [18]:
from dataclasses import asdict
from pathlib import Path
from llm_training.config import TrainConfig

In [17]:
import sys
sys.path.append('../src/')

In [19]:
project_root = config_path.parent.parent

In [20]:
print(project_root)

..


In [21]:
data_path = project_root / config["data"]["path"]
output_dir = project_root / config["output"]["directory"]

In [23]:
print(output_dir)

../models/adapter_experiment


In [24]:
print("Data path:", data_path)
print("Exists:", data_path.exists())
print("Output directory:", output_dir)

Data path: ../data/01_cold_start_cot_sft/data.jsonl
Exists: True
Output directory: ../models/adapter_experiment


## Building TrainConfig class instance

In [25]:
train_config = TrainConfig(
    model_name=config["model"]["name"],
    data_path=data_path,
    output_dir=output_dir,
    lora_r=config["peft"]["r"],
    lora_alpha=config["peft"]["alpha"],
    lora_dropout=config["peft"]["dropout"],
    lora_target_modules=tuple(config["peft"]["target_modules"]),
    num_train_epochs=config["training"]["num_train_epochs"],
    per_device_train_batch_size=config["training"]["batch_size"],
    gradient_accumulation_steps=config["training"]["gradient_accumulation_steps"],
    learning_rate=config["training"]["learning_rate"],
    max_length=config["training"]["max_length"],
    seed=config["training"]["seed"],
)

In [26]:
for name, value in asdict(train_config).items():
    print(f"{name}: {value!r}")

model_name: 'Qwen/Qwen2.5-1.5B'
data_path: PosixPath('../data/01_cold_start_cot_sft/data.jsonl')
output_dir: PosixPath('../models/adapter_experiment')
lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target_modules: ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
num_train_epochs: 3
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 0.1002
max_length: 1024
warmup_ratio: 0.03
lr_scheduler_type: 'cosine'
logging_steps: 10
eval_steps: 50
save_steps: 200
save_total_limit: 2
max_steps: None
fp16: False
bf16: True
seed: 42
report_to: 'none'


# Connecting the config to messages now 

In [27]:
from llm_training.data.messages import (
    load_jsonl,
    to_message,
    to_sft_dataset,
)

## Loadigng and inspecting records

In [28]:
records = load_jsonl(train_config.data_path)

In [30]:
print(records[0])

{'spec': {'topic': 'damaged-item', 'tone': 'confused', 'item': 'running sneakers', 'days_since_purchase': 6, 'payment_method': 'gift card', 'order_id': 'ORD-201414'}, 'prompt': "Hi, I just opened my order of running sneakers (ORD-201414) and there's a big tear in the side. I'm not sure what to do – can I return them? I paid with a gift card and I'm worried I'll lose the money.", 'response': "<thinking>The customer is reporting a damaged item (running sneakers) from order ORD-201414, which was purchased 6 days ago. According to policy, the standard return window is 30 days from delivery date, and this order is within that window. Additionally, damaged or wrong items are always refundable with a free return label, even outside the 30-day window. Since the item is damaged, we can authorize a return. The original payment method was a gift card, so the refund will go back to the same gift card within 5-7 business days after the return is processed. The customer is confused and worried about

In [32]:
assert "prompt" in records[10]
assert "response" in records[10]

## Converting one record (passing through our messages)

In [34]:
message_record = to_message(records[0])

In [36]:
message_record

{'messages': [{'role': 'user',
   'content': "Hi, I just opened my order of running sneakers (ORD-201414) and there's a big tear in the side. I'm not sure what to do – can I return them? I paid with a gift card and I'm worried I'll lose the money."},
  {'role': 'assistant',
   'content': "<thinking>The customer is reporting a damaged item (running sneakers) from order ORD-201414, which was purchased 6 days ago. According to policy, the standard return window is 30 days from delivery date, and this order is within that window. Additionally, damaged or wrong items are always refundable with a free return label, even outside the 30-day window. Since the item is damaged, we can authorize a return. The original payment method was a gift card, so the refund will go back to the same gift card within 5-7 business days after the return is processed. The customer is confused and worried about losing the money, so we need to reassure them and explain the process clearly. We'll provide a free retu

## Creating the HF dataset

In [39]:
train_ds, eval_ds = to_sft_dataset(
    train_config.data_path,
    val_frac=0.1,
    seed=train_config.seed,
)

In [40]:
print("Train rows:", len(train_ds))
print("Eval rows:", len(eval_ds))
print("Columns:", train_ds.column_names)


Train rows: 446
Eval rows: 50
Columns: ['spec', 'prompt', 'response', 'messages']


In [41]:
train_ds[0]

{'spec': {'topic': 'exchange-size',
  'tone': 'urgent',
  'item': 'kids pajamas',
  'days_since_purchase': 41,
  'payment_method': 'gift card',
  'order_id': 'ORD-667296'},
 'prompt': "I need to exchange the kids pajamas I bought from order ORD-667296 for a different size. They are too small. I bought them 41 days ago and I'm in a hurry because my child needs them for a school event. Can you help me with an exchange or refund?",
 'response': "<thinking>Customer is requesting an exchange for a different size due to fit issue. Item was purchased 41 days ago, which is beyond the 30-day return window from delivery date. Payment was made via gift card on order ORD-667296. Policy states that standard returns/exchanges are only accepted within 30 days of delivery. The item is not damaged or wrong (size issue is not covered under the exception for damaged or wrong items). Therefore, we cannot process a return or exchange under the current policy. Since the customer is urgent, we should apologi

## Inspecting model settings

In [42]:
model_name = config["model"]["name"]

In [43]:
model_name

'Qwen/Qwen2.5-1.5B'

In [44]:
peft_config = config["peft"]

In [45]:
peft_config

{'method': 'lora',
 'r': 16,
 'alpha': 32,
 'dropout': 0.05,
 'target_modules': ['q_proj',
  'k_proj',
  'v_proj',
  'o_proj',
  'gate_proj',
  'up_proj',
  'down_proj']}

## Build LoRA configuration

In [48]:
from peft import LoraConfig
if peft_config["method"] != "lora":
    raise NotImplementedError(
        f"PEFT method not implemented yet: {peft_config['method']}"
    )

In [49]:
lora_config = LoraConfig(
    r=peft_config["r"],
    lora_alpha=peft_config["alpha"],
    lora_dropout=peft_config["dropout"],
    target_modules=peft_config["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)

## Inspect the generated PEFT Obect

In [51]:
print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'up_proj', 'down_proj', 'q_proj', 'v_proj', 'gate_proj', 'k_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [52]:
assert lora_config.r == config["peft"]["r"]
assert lora_config.lora_alpha == config["peft"]["alpha"]
assert lora_config.lora_dropout == config["peft"]["dropout"]
assert lora_config.target_modules == set(config["peft"]["target_modules"])


## Testing invalid method handling

In [53]:
test_method = "ia3"
if test_method != "lora":
    print(f"Would require a different PEFT config: {test_method}")

Would require a different PEFT config: ia3


### The proves that different PEFT algorithms do not silently receive LoRA-specific parameters

# Loading Tokenizer + Model Configuration (without the model weights)

In [54]:
from transformers import AutoConfig, AutoTokenizer

In [55]:
model_name = train_config.model_name

In [58]:
model_name

'Qwen/Qwen2.5-1.5B'

In [56]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_architecture = AutoConfig.from_pretrained(model_name)

In [57]:
print("Tokenizer vocabulary size:", tokenizer.vocab_size)
print("Model type:", model_architecture.model_type)
print("Hidden size:", getattr(model_architecture, "hidden_size", None))
print(
    "Number of layers:",
    getattr(model_architecture, "num_hidden_layers", None),
)
print("Number of attention heads:",
      getattr(model_architecture, "num_attention_heads", None))

Tokenizer vocabulary size: 151643
Model type: qwen2
Hidden size: 1536
Number of layers: 28
Number of attention heads: 12


In [59]:
sample_text = records[0]["prompt"]
sample_text

"Hi, I just opened my order of running sneakers (ORD-201414) and there's a big tear in the side. I'm not sure what to do – can I return them? I paid with a gift card and I'm worried I'll lose the money."

In [60]:
tokens = tokenizer(sample_text)

In [61]:
tokens

{'input_ids': [13048, 11, 358, 1101, 8930, 847, 1973, 315, 4303, 67191, 320, 4276, 12, 17, 15, 16, 19, 16, 19, 8, 323, 1052, 594, 264, 2409, 17576, 304, 279, 3108, 13, 358, 2776, 537, 2704, 1128, 311, 653, 1365, 646, 358, 470, 1105, 30, 358, 7171, 448, 264, 8189, 3701, 323, 358, 2776, 17811, 358, 3278, 9052, 279, 3220, 13], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [62]:
print("Token IDs:", tokens["input_ids"][:20])
print("Token count:", len(tokens["input_ids"]))
print("Decoded text:")
print(tokenizer.decode(tokens["input_ids"]))

Token IDs: [13048, 11, 358, 1101, 8930, 847, 1973, 315, 4303, 67191, 320, 4276, 12, 17, 15, 16, 19, 16, 19, 8]
Token count: 59
Decoded text:
Hi, I just opened my order of running sneakers (ORD-201414) and there's a big tear in the side. I'm not sure what to do – can I return them? I paid with a gift card and I'm worried I'll lose the money.


## Comparing token lenght against above config

In [64]:
token_count = len(tokens["input_ids"])

print("Token count:", token_count)
print("Configured max length:", train_config.max_length)



Token count: 59
Configured max length: 1024


In [65]:
if token_count > train_config.max_length:
    print("Warning: this example exceeds max_length.")
else:
    print("Example fits within max_length.")

Example fits within max_length.


## Inspecting model architecture

In [66]:
model_architecture

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 131072,
  "max_window_layers": 28,
  "model_type": "qwen2",
 

# Loading model weights and investigating the available layers that we are performing lora on

In [67]:
from transformers import AutoModelForCausalLM

In [68]:
model = AutoModelForCausalLM.from_pretrained(
    train_config.model_name, 
    torch_dtype="auto",
    low_cpu_mem_usage=True,
)

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████| 338/338 [00:00<00:00, 5114.56it/s]


In [69]:
target_modules = config["peft"]["target_modules"]

In [70]:
target_modules

['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

In [71]:
module_matches = {
    target: []
    for target in target_modules
}

In [72]:
model.named_modules

<bound method Module.named_modules of Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2

In [73]:
for module_name, module in model.named_modules():
    leaf_name = module_name.rsplit(".", maxsplit=1)[-1]

    if leaf_name in module_matches:
        module_matches[leaf_name].append(
            (module_name, type(module).__name__)
        )

In [74]:
for target, matches in module_matches.items():
    print(f"\n{target}: {len(matches)} matches")

    for module_name, module_type in matches[:5]:
        print(f"  {module_name} ({module_type})")


q_proj: 28 matches
  model.layers.0.self_attn.q_proj (Linear)
  model.layers.1.self_attn.q_proj (Linear)
  model.layers.2.self_attn.q_proj (Linear)
  model.layers.3.self_attn.q_proj (Linear)
  model.layers.4.self_attn.q_proj (Linear)

k_proj: 28 matches
  model.layers.0.self_attn.k_proj (Linear)
  model.layers.1.self_attn.k_proj (Linear)
  model.layers.2.self_attn.k_proj (Linear)
  model.layers.3.self_attn.k_proj (Linear)
  model.layers.4.self_attn.k_proj (Linear)

v_proj: 28 matches
  model.layers.0.self_attn.v_proj (Linear)
  model.layers.1.self_attn.v_proj (Linear)
  model.layers.2.self_attn.v_proj (Linear)
  model.layers.3.self_attn.v_proj (Linear)
  model.layers.4.self_attn.v_proj (Linear)

o_proj: 28 matches
  model.layers.0.self_attn.o_proj (Linear)
  model.layers.1.self_attn.o_proj (Linear)
  model.layers.2.self_attn.o_proj (Linear)
  model.layers.3.self_attn.o_proj (Linear)
  model.layers.4.self_attn.o_proj (Linear)

gate_proj: 28 matches
  model.layers.0.mlp.gate_proj (Linea

In [75]:
missing_targets = [
    target
    for target, matches in module_matches.items()
    if not matches
]

if missing_targets:
    raise ValueError(
        f"These target modules were not found: {missing_targets}"
    )

print("All configured target modules exist in the model.")

All configured target modules exist in the model.


# Connecting LoRA config to the loaded model

In [76]:
from peft import get_peft_model

In [77]:
peft_model = get_peft_model(model, lora_config)

## Inspect trainable parameters

In [79]:
peft_model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [81]:
trainable_parameters = [
    (name, parameter)
    for name, parameter in peft_model.named_parameters()
    if parameter.requires_grad
]

In [82]:
frozen_parameters = [
    (name, parameter)
    for name, parameter in peft_model.named_parameters()
    if not parameter.requires_grad
]

In [83]:
print("Trainable tensors:", len(trainable_parameters))
print("Frozen tensors:", len(frozen_parameters))

Trainable tensors: 392
Frozen tensors: 338


In [84]:
assert trainable_parameters
assert frozen_parameters

## Inspecting trainable names

In [85]:
for name, _ in trainable_parameters[:20]:
    print(name)

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.up_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight
base_model.model.model.layer

# Running one forward pass

In [87]:
import torch

In [88]:
device = next(peft_model.parameters()).device

In [90]:
inputs = tokenizer(
    records[0]["prompt"],
    return_tensors="pt",
)

In [91]:
inputs

{'input_ids': tensor([[13048,    11,   358,  1101,  8930,   847,  1973,   315,  4303, 67191,
           320,  4276,    12,    17,    15,    16,    19,    16,    19,     8,
           323,  1052,   594,   264,  2409, 17576,   304,   279,  3108,    13,
           358,  2776,   537,  2704,  1128,   311,   653,  1365,   646,   358,
           470,  1105,    30,   358,  7171,   448,   264,  8189,  3701,   323,
           358,  2776, 17811,   358,  3278,  9052,   279,  3220,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [92]:
inputs = {
    name: value.to(device)
    for name, value in inputs.items()
}

In [93]:
inputs

{'input_ids': tensor([[13048,    11,   358,  1101,  8930,   847,  1973,   315,  4303, 67191,
            320,  4276,    12,    17,    15,    16,    19,    16,    19,     8,
            323,  1052,   594,   264,  2409, 17576,   304,   279,  3108,    13,
            358,  2776,   537,  2704,  1128,   311,   653,  1365,   646,   358,
            470,  1105,    30,   358,  7171,   448,   264,  8189,  3701,   323,
            358,  2776, 17811,   358,  3278,  9052,   279,  3220,    13]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [94]:
with torch.no_grad():
    outputs = peft_model(**inputs)

In [95]:
print("Logits shape:", outputs.logits.shape)

Logits shape: torch.Size([1, 59, 151936])


## Creating a tiny dataset for a training run

In [96]:
small_train_ds = train_ds.select(
    range(min(4, len(train_ds)))
)

In [97]:
small_eval_ds = eval_ds.select(
    range(min(2, len(eval_ds)))
)

In [98]:
print("Tiny train size:", len(small_train_ds))
print("Tiny eval size:", len(small_eval_ds))
print(small_train_ds[0])

Tiny train size: 4
Tiny eval size: 2
{'spec': {'topic': 'exchange-size', 'tone': 'urgent', 'item': 'kids pajamas', 'days_since_purchase': 41, 'payment_method': 'gift card', 'order_id': 'ORD-667296'}, 'prompt': "I need to exchange the kids pajamas I bought from order ORD-667296 for a different size. They are too small. I bought them 41 days ago and I'm in a hurry because my child needs them for a school event. Can you help me with an exchange or refund?", 'response': "<thinking>Customer is requesting an exchange for a different size due to fit issue. Item was purchased 41 days ago, which is beyond the 30-day return window from delivery date. Payment was made via gift card on order ORD-667296. Policy states that standard returns/exchanges are only accepted within 30 days of delivery. The item is not damaged or wrong (size issue is not covered under the exception for damaged or wrong items). Therefore, we cannot process a return or exchange under the current policy. Since the customer is 

In [99]:
train_for_smoke = small_train_ds.remove_columns(
    ["spec", "prompt", "response"]
)


In [100]:
eval_for_smoke = small_eval_ds.remove_columns(
    ["spec", "prompt", "response"]
)


In [101]:
print(train_for_smoke.column_names)

['messages']


In [102]:
from trl import SFTConfig, SFTTrainer

In [104]:
if tokenizer.pad_token is None: 
    tokenizer.pad_token = tokenizer.eos_token

In [105]:
peft_model.config.use_cache = False

In [106]:
smoke_args = SFTConfig(
    output_dir="models/smoke_test",
    max_length=train_config.max_length,
    packing=False,
    completion_only_loss=True,
    max_steps=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=train_config.learning_rate,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=False,
    bf16=False,
)

In [107]:
trainer = SFTTrainer(
    model=peft_model,
    processing_class=tokenizer,
    args=smoke_args,
    train_dataset=train_for_smoke,
    eval_dataset=eval_for_smoke,
)

Truncating train dataset: 100%|█████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 892.03 examples/s]
Dropping fully masked examples from train dataset: 100%|███████████████████████████████████████████████████| 4/4 [00:00<00:00, 1152.44 examples/s]
Dropping fully masked examples from eval dataset: 100%|█████████████████████████████████████████████████████| 2/2 [00:00<00:00, 713.26 examples/s]


In [108]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,2.164708,11.514119,5.432991,468.000000,0.011799


TrainOutput(global_step=1, training_loss=2.164707660675049, metrics={'train_runtime': 11.5411, 'train_samples_per_second': 0.087, 'train_steps_per_second': 0.087, 'total_flos': 3731285495808.0, 'train_loss': 2.164707660675049, 'epoch': 0.25})